# Genetic Algorithms and OSGA in the Prompt Embedding Space
Notebook Version: 0.11.0 (06/10/2026)
* Bounded OSGA, fixed operator pools and complete attempt/lineage export.
Initial operator-pool implementation, copied from the current GA notebook.

Configure first; the single run cell below is the only optimization entry point.
This notebook has not been run as an experiment.


## Working checkout setup (local or Colab)


In [ ]:
from pathlib import Path
import sys
import subprocess

checkout = Path.cwd()
if not (checkout / "setup.py").exists():
    checkout = checkout.parent  # Notebook opened from notebooks/.
if not (checkout / "evolutionary" / "algorithms" / "ga.py").exists():
    raise RuntimeError("Set checkout to your working checkout before installing")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
                       f"{checkout}[prompt_embedding,imaging]"])


In [ ]:
# Mount drive to save results
from google.colab import drive
import evolutionary_imaging.processing as ip
import evolutionary_prompt_embedding.tensorboard_embed_visualizer as ev
drive.mount("/content/drive")
base_path = "/content/drive/MyDrive/evolutionary/"
ip.RESULTS_FOLDER = base_path + ip.RESULTS_FOLDER
ev.DEFAULT_OUTPUT_FOLDER = base_path + "vis"
save_run_path = base_path + "saved_runs"

In [ ]:
# Check if GPU is available
import torch
print(torch.cuda.is_available())

## Project Setup

In [ ]:
from evolutionary.plotting import plot_fitness_statistics, plot_time_statistics
import evolutionary_imaging.processing as ip
import evolutionary_prompt_embedding.tensorboard_embed_visualizer as ev
from diffusers.utils import logging
from evolutionary_imaging.processing import create_animation_from_generations, create_generation_image_grid, save_images_from_generation
import torch
import os

In [ ]:
logging.disable_progress_bar()
logging.set_verbosity_error()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
use_visualizer = True
save_images = True
# Optional: override ip.RESULTS_FOLDER, ev.DEFAULT_OUTPUT_FOLDER, save_run_path.


### Configuration and interpretation
OSGA requires `ceil(success_ratio * population_size)` successful children in a full offspring population.
Here 0.6 permits up to 40 unsuccessful selected children before elitism. The comparison factor 1.0
requires strict improvement over the better parent; equality fails. Pressure 10.0 permits at most
1,000 evaluated children per generation. Elitism then replaces the worst selected child.
An unmet quota stops the run without committing a partial population; the returned best candidate
can come from that interrupted generation. Its image need not appear in completed-population grids.

Mutation event probability 0.2 decides whether to invoke a mutator. Gaussian/polynomial/donor
coordinate or row fractions decide what changes inside that event. Spherical mutation uses rows,
including the whole pooled vector. Fixed equal operator weights do not adapt over time.
SLERP and spherical mutation preserve/interpolate row norms before bounds repair; repair may change norms.

Ordinary GA (`offspring_selection = None`) uses **9,901 evaluations**: 100 initial candidates plus
99 generations with 99 new candidates and one cached elite. OSGA creates 100 offspring before
elitism and may reject more. The common 10,000 cap is a fair maximum; report actual evaluation costs
and completed generations in later comparisons. No experiments or score improvement are claimed here.

Raw records include every evaluated candidate; survivor keys exclude rejected, displaced and
uncommitted children. Pair summaries report successes/attempts and final survivors/attempts.
Operator outcomes are outcomes of combinations, not isolated causal effects. Generations and
evaluation indices support later phase analysis without defining phases now.

Models load before seeding so cold and cached setup produce the same variation stream.
Save the notebook before running and keep the source unchanged through export. JSON includes
tracked changes and new implementation source; commit the feature before published experiments.


In [ ]:
import random
import numpy as np
from dataclasses import asdict
from evolutionary_prompt_embedding.argument_types import PooledPromptEmbedData
from evolutionary_prompt_embedding.image_creation import SDXLPromptEmbeddingImageCreator
from evolutionary_prompt_embedding.variation import (
    UniformGaussianMutatorArguments, PooledUniformGaussianMutator,
    PooledUniformCrossover, PooledRowUniformCrossover, PooledOnePointCrossover,
    PooledTwoPointCrossover, PooledArithmeticCrossover, PooledSlerpCrossover,
    PooledBLXAlphaCrossover, PooledSBXCrossover, PooledSphericalRotationMutator,
    PooledDonorReplacementMutator, PooledPolynomialMutator,
)
from evolutionary.operators import MultiCrossover, MultiMutator
from evolutionary_prompt_embedding.value_ranges import SDXLTurboEmbeddingRange, SDXLTurboPooledEmbeddingRange
from evolutionary.evolutionary_selectors import TournamentSelector
from evolutionary.algorithms.ga import GeneticAlgorithm, OffspringSelectionConfig
from evolutionary_imaging.evaluators import AestheticsImageEvaluator, MultiCLIPIQAEvaluator
from evolutionary_prompt_embedding.tensorboard_embed_visualizer import TensorboardEmbedVisualizer, EmbeddingVariant

population_size = 100
num_generations = 100  # Includes initialization as generation 0.
elitism = 1
tournament_size = 3
batch_size = 1
inference_steps = 3
crossover_rate = 0.9
mutation_rate = 0.2  # Probability of a mutation EVENT, independent of coordinate/row participation.
prompt = None
seed = 42
fixed_noise_seeds = [0]
offspring_selection = OffspringSelectionConfig(0.6, 1.0, 10.0)
# Use None for ordinary GA, keeping all other operators, seeds and the cap the same.
max_evaluations = 10_000

creator = SDXLPromptEmbeddingImageCreator(batch_size=batch_size, inference_steps=inference_steps,
                                         fixed_noise_seeds=fixed_noise_seeds)
evaluator = AestheticsImageEvaluator()  # LAION improved aesthetic predictor V2.

# Seed after model setup: cold and cached model loading consume different RNG draws.
# Diffusion uses fresh separate generators for every candidate and retry.
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

embedding_range = SDXLTurboEmbeddingRange()
pooled_embedding_range = SDXLTurboPooledEmbeddingRange()
bounds = (embedding_range.minimum, embedding_range.maximum)
pooled_bounds = (pooled_embedding_range.minimum, pooled_embedding_range.maximum)
init_args = [PooledPromptEmbedData(embedding_range.random_tensor_in_range(),
                                 pooled_embedding_range.random_tensor_in_range())
             for _ in range(population_size)]

operator_parameters = {
    "coordinate_uniform": {"swap_rate": 0.5, "swap_rate_pooled": 0.5},
    "row_uniform": {"swap_rate": 0.5},
    "one_point": {"prompt_axis": "row", "pooled_axis": "coordinate"},
    "two_point": {"prompt_axis": "row", "pooled_axis": "coordinate"},
    "arithmetic": {"interpolation_weight": 0.5, "interpolation_weight_pooled": 0.5,
                   "proportion": 1.0, "proportion_pooled": 1.0},
    "slerp": {"ratio": 0.5, "ratio_pooled": 0.5},
    "blx_alpha": {"alpha": 0.5},
    "sbx": {"index": 15, "participation": 0.5},
    "gaussian": {"prompt": {"mutation_rate": 0.2, "mutation_strength": 2.0},
                 "pooled": {"mutation_rate": 0.2, "mutation_strength": 0.4}},
    "spherical": {"mutation_rate": 0.1, "mutation_angle": 0.1,
                  "mutation_rate_pooled": 0.1, "mutation_angle_pooled": 0.1},
    "donor": {"participation": 0.1, "bank": "frozen initial population"},
    "polynomial": {"index": 20, "participation": 0.1},
}
crossover = MultiCrossover({
    "coordinate_uniform": PooledUniformCrossover(**operator_parameters["coordinate_uniform"]),
    "row_uniform": PooledRowUniformCrossover(**operator_parameters["row_uniform"]),
    "one_point": PooledOnePointCrossover(),
    "two_point": PooledTwoPointCrossover(),
    "arithmetic": PooledArithmeticCrossover(**operator_parameters["arithmetic"]),
    "slerp": PooledSlerpCrossover(clamp_range=bounds, clamp_range_pooled=pooled_bounds,
                                    **operator_parameters["slerp"]),
    "blx_alpha": PooledBLXAlphaCrossover(bounds, pooled_bounds, **operator_parameters["blx_alpha"]),
    "sbx": PooledSBXCrossover(bounds, pooled_bounds, **operator_parameters["sbx"]),
})
mutator = MultiMutator({
    "gaussian": PooledUniformGaussianMutator(
        UniformGaussianMutatorArguments(clamp_range=bounds, **operator_parameters["gaussian"]["prompt"]),
        UniformGaussianMutatorArguments(clamp_range=pooled_bounds, **operator_parameters["gaussian"]["pooled"])),
    "spherical": PooledSphericalRotationMutator(clamp_range=bounds, clamp_range_pooled=pooled_bounds,
                                                 **operator_parameters["spherical"]),
    "donor": PooledDonorReplacementMutator(init_args, operator_parameters["donor"]["participation"],
                                            bounds, pooled_bounds),
    "polynomial": PooledPolynomialMutator(bounds, pooled_bounds, **operator_parameters["polynomial"]),
})
selector = TournamentSelector(tournament_size=tournament_size)
metrics = ("natural", "quality", "contrast", "sharpness", "noisiness")
visualizer = TensorboardEmbedVisualizer[PooledPromptEmbedData, list[str]](["Index", "Generation", "Fitness"])

def post_evaluation_callback(g, a):
    image_paths = save_images_from_generation(a.population, g) if save_images else None
    if use_visualizer:
        for i, solution in enumerate(a.population):
            visualizer.add_embedding(solution.arguments, [str(i), str(g), f"{solution.fitness:.3f}"],
                                     image_paths[i] if image_paths else None)

ga = GeneticAlgorithm(
    population_size=population_size, num_generations=num_generations,
    solution_creator=creator, evaluator=evaluator, mutator=mutator, crossover=crossover,
    selector=selector, initial_arguments=init_args, elitism_count=elitism,
    crossover_rate=crossover_rate, mutation_rate=mutation_rate,
    offspring_selection=offspring_selection, max_evaluations=max_evaluations,
    post_evaluation_callback=post_evaluation_callback,
)
run_configuration = {
    "population_size": population_size, "num_generations": num_generations,
    "elitism_count": elitism, "tournament_size": tournament_size,
    "batch_size": batch_size, "inference_steps": inference_steps,
    "model_id": "stabilityai/sdxl-turbo", "evaluator": "LAION improved aesthetic predictor V2",
    "initialization": "random embeddings", "prompt": prompt,
    "crossover_rate": crossover_rate, "mutation_event_probability": mutation_rate,
    "offspring_selection": asdict(offspring_selection) if offspring_selection is not None else None,
    "max_evaluations": max_evaluations, "operator_parameters": operator_parameters,
    "crossover_weights": dict(zip(crossover.operators, crossover.weights)),
    "mutation_weights": dict(zip(mutator.operators, mutator.weights)),
    "bounds": {"prompt": [float(v) for v in bounds], "pooled": [float(v) for v in pooled_bounds]},
    "seeds": {"python": seed, "numpy": seed, "torch": seed, "diffusion": fixed_noise_seeds},
}


In [ ]:
# Explicit optimization entry point. Execute only when ready for an experiment.
best_solution = ga.run()
print(ga.termination_reason, ga.evaluation_count, ga.completed_generations)


In [ ]:
from diffusers.utils import make_image_grid

# Show best solution
print(best_solution.fitness)
make_image_grid(best_solution.result.images, 1, batch_size)

## Visualize Embeddings with the Tensorboard Embedding Projector

In [ ]:
# This will save the embeddings and the metadata to your disk
visualizer.generate_visualization(
    sprite_single_image_dim=(60, 60),
    include_variants=(EmbeddingVariant.NORMAL, EmbeddingVariant.COMBINED_APPEND),
    # filter_predicate=lambda e, l, i: int(l[0]) < 27 #or float(l[2]) > 8.0, # Adjust this to filter embeddings if needed
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir={visualizer.output_folder}

## Compare results to their prompt (only for CLIP-Score, AI-Detection)

In [ ]:
from evolutionary_prompt_embedding.image_creation import SDXLPromptEmbeddingImageCreator
creator_compare = (SDXLPromptEmbeddingImageCreator(batch_size=4, inference_steps=inference_steps,
                                                  fixed_noise_seeds=[0, 1, 2, 3])
                   if prompt is not None else None)


In [ ]:
from diffusers.utils import make_image_grid
if prompt is not None:
    args = creator_compare.arguments_from_prompt(prompt)
    solution = creator_compare.create_solution(args)
    print(evaluator.evaluate(solution.result))
    display(make_image_grid(solution.result.images, 2, 2))
else:
    print("Prompt comparison skipped: initialization used random embeddings.")


## Create a video from the generational progress showing the top images

In [ ]:
for gen in range(ga.completed_generations):
    create_generation_image_grid(gen, max_images=10)

In [ ]:
video_loc = create_animation_from_generations(ga.completed_generations)
print(video_loc)

## Plot statistics

In [ ]:
stats = ga.statistics
plot_fitness_statistics(ga.completed_generations, stats.best_fitness, stats.worst_fitness, stats.avg_fitness).savefig("results/fitness_statistics.pdf")

In [ ]:
plot_time_statistics(stats.evaluation_time, stats.creation_time, stats.post_evaluation_time).savefig("results/time_statistics.pdf")

In [ ]:
stats.avg_fitness_stats()

## Multiobjective Stats
Read the SO results from disk and evaluate with the MO evaluator to create a stats object.
Its not as efficient as directly saving the MO stats during the run, but it works as a fallback.
The GA does not know about MultiObjectiveFitness so it only saves single value stats.

In [ ]:
from evolutionary.statistics import StatisticsTracker
from PIL import Image
from evolutionary.evolution_base import SolutionCandidate
from evolutionary_imaging.image_base import ImageSolutionData
import glob
import gc
import pickle
from tqdm import tqdm

clip_iqa_evaluator = MultiCLIPIQAEvaluator(metrics=metrics, clip_model="clip_iqa")
mo_stats = StatisticsTracker()
stats_file = "results/mo_stats.pkl"

for gen in tqdm(range(ga.completed_generations), desc="Evaluating generations"):
    generation_dir = os.path.join(ip.RESULTS_FOLDER, f"{gen}")
    image_files = [f for f in glob.glob(os.path.join(generation_dir, "*.png")) if "fitness" in os.path.basename(f)]

    if not image_files:
        raise RuntimeError(f"No saved images for completed generation {gen}")
    population = []
    for img_path in image_files:
        img = Image.open(img_path)
        solution_data = ImageSolutionData(images=[img])
        fitness = clip_iqa_evaluator.evaluate(solution_data)
        solution = SolutionCandidate(arguments=None, result=solution_data)
        solution.fitness = fitness
        population.append(solution)
    mo_stats.update_fitness(population)
    del population
    gc.collect()

    with open(stats_file, "wb") as f:
        pickle.dump(mo_stats, f)


In [ ]:
mo_stats.avg_fitness_stats()

In [ ]:
fig = plot_fitness_statistics(len(mo_stats.best_fitness), best_fitness=mo_stats.best_fitness, labels=[m if isinstance(m, str) else " ".join(m) for m in metrics])
fig.savefig('results/best_fitness_statistics.pdf')

In [ ]:
fig = plot_fitness_statistics(len(mo_stats.best_fitness), avg_fitness=mo_stats.avg_fitness, labels=[m if isinstance(m, str) else " ".join(m) for m in metrics])
fig.savefig('results/avg_fitness_statistics.pdf')

## Save notebook and components

In [ ]:
!jupyter nbconvert --to html ga_osga_notebook.ipynb


### Save the run to disk

In [ ]:
import pickle
import json
import os
import platform
from dataclasses import asdict
from datetime import datetime
from importlib.metadata import version, PackageNotFoundError
import subprocess

if "save_run_path" not in globals():
    save_run_path = "saved_runs"
os.makedirs(save_run_path, exist_ok=True)
output_stem = os.path.join(save_run_path, f"ga_osga_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")
with open(output_stem + ".pkl", "wb") as f:
    pickle.dump(ga, f)

versions = {"python": platform.python_version()}
for package in ("evolutionary", "torch", "numpy", "diffusers", "transformers", "accelerate",
                "clip", "torchvision", "Pillow", "pytorch-lightning", "tensorboard"):
    try:
        versions[package] = version(package)
    except PackageNotFoundError:
        versions[package] = None
revision = subprocess.run(["git", "rev-parse", "HEAD"], cwd=checkout, text=True,
                          capture_output=True, check=True).stdout.strip()
working_tree_diff = subprocess.run(["git", "diff", "HEAD"], cwd=checkout, text=True,
                                   capture_output=True, check=True).stdout
# The revision/diff cover tracked source; include new implementation files too.
untracked_paths = subprocess.run(["git", "ls-files", "--others", "--exclude-standard", "-z"],
                                 cwd=checkout, text=True, capture_output=True, check=True).stdout.split("\0")
untracked_sources = {
    path: (checkout / path).read_text()
    for path in untracked_paths
    if (path.endswith(".py") and path.startswith(("evolutionary/", "evolutionary_")))
    or path == "notebooks/ga_osga_notebook.ipynb"
}
stats = ga.statistics
export_data = {
    "configuration": run_configuration, "versions": versions, "git_revision": revision,
    "working_tree_diff": working_tree_diff, "untracked_sources": untracked_sources,
    "device": str(init_args[0].prompt_embeds.device), "dtype": str(init_args[0].prompt_embeds.dtype),
    "actual_evaluation_count": ga.evaluation_count, "termination_reason": ga.termination_reason,
    "completed_generations": ga.completed_generations,
    "evaluations": [asdict(record) for record in stats.evaluation_records],
    "generation_summaries": [asdict(summary) for summary in stats.generation_summaries],
    "lineage": [asdict(item) for item in stats.solution_history.values()],
    "operator_summary": stats.operator_summary(),
    "statistics": {"best_fitness": stats.best_fitness, "avg_fitness": stats.avg_fitness,
                   "worst_fitness": stats.worst_fitness, "creation_time": stats.creation_time,
                   "evaluation_time": stats.evaluation_time, "post_evaluation_time": stats.post_evaluation_time},
}
with open(output_stem + ".json", "w") as f:
    json.dump(export_data, f, indent=2, allow_nan=False)
print(f"Run saved to {output_stem}.pkl and .json")


### Load the run from disk 
Notebook and library versions should match with the saved run

In [ ]:
import pickle
import os

with open(os.path.join("saved_runs", "insert_filename"), "rb") as f:
    run = pickle.load(f)

## Fallback functions for when something went wrong

### Access Best Solution from Disk

In [ ]:
import os
import glob
import evolutionary_imaging.processing as ip
from PIL import Image
from evolutionary_imaging.image_base import ImageSolutionData

num_generations = ga.completed_generations # Set this to the number of generations you ran (if you didn't finish)
generation_dir = os.path.join(ip.RESULTS_FOLDER, f"{ga.completed_generations - 1}")
image_files = glob.glob(os.path.join(generation_dir, "*.png"))
image_files.sort(key=ip.fitness_filename_sorting_key, reverse=True)
print(image_files[0])
Image.open(image_files[0])
clip_iqa_evaluator.evaluate(ImageSolutionData(images=[Image.open(image_files[0])]))

### ffmpeg is not installed, create GIF instead

In [ ]:
from evolutionary_imaging.processing import create_animation_from_generations_pil
video_loc = create_animation_from_generations_pil(ga.completed_generations)
print(video_loc)